# Corrected guidance wrapper

Reference implementation of the fixed guidance wrapper + equivariant affinity model that keeps pocket atoms batched correctly and centered in the same frame as TargetDiff sampling.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.utils import scatter
from e3nn import o3
from e3nn.nn import Gate
import utils.transforms as trans


In [ ]:
def build_radius_edge_index(pos: torch.Tensor, batch: torch.Tensor = None, r: float = 5.0, loop: bool = False):
    """
    Pure PyTorch radius graph that respects batch.
    pos: [N, 3]
    batch: [N] or None
    returns edge_index: [2, E]
    """
    if batch is None:
        batch = torch.zeros(pos.size(0), dtype=torch.long, device=pos.device)

    edge_src = []
    edge_dst = []

    for b in batch.unique():
        mask = batch == b
        idx = mask.nonzero(as_tuple=True)[0]
        sub_pos = pos[idx]

        dist = torch.cdist(sub_pos, sub_pos, p=2)
        sub_mask = dist <= r
        if not loop:
            sub_mask = sub_mask & ~torch.eye(len(idx), dtype=torch.bool, device=pos.device)

        src, dst = sub_mask.nonzero(as_tuple=True)
        edge_src.append(idx[src])
        edge_dst.append(idx[dst])

    if len(edge_src) == 0:
        return torch.empty(2, 0, dtype=torch.long, device=pos.device)
    edge_index = torch.stack([torch.cat(edge_src), torch.cat(edge_dst)], dim=0)
    return edge_index


In [ ]:
# Node feature irreps
IRREPS_NODE   = o3.Irreps("32x0e + 32x1o")
IRREPS_GATE_IN = o3.Irreps("32x0e + 32x0e + 32x1o")
IRREPS_OUT    = o3.Irreps("32x0e")
IRREPS_SH     = o3.Irreps("1x1o")

HIDDEN = 128
DROPOUT = 0.1

class EquivariantMPBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.lin_node = o3.Linear(
            irreps_in=IRREPS_NODE,
            irreps_out=IRREPS_GATE_IN,
        )

        self.gate = Gate(
            irreps_scalars="32x0e",
            act_scalars=[F.relu],
            irreps_gates="32x0e",
            act_gates=[torch.sigmoid],
            irreps_gated="32x1o",
        )

        self.tp_msg = o3.FullyConnectedTensorProduct(
            irreps_in1=IRREPS_NODE,
            irreps_in2=IRREPS_SH,
            irreps_out=IRREPS_NODE,
        )

        self.lin_msg = o3.Linear(
            irreps_in=IRREPS_NODE,
            irreps_out=IRREPS_NODE,
        )

        self.norm = nn.LayerNorm(IRREPS_NODE.dim)

    def forward(self, x, pos, edge_index):
        row, col = edge_index

        h = self.lin_node(x)
        h = self.gate(h)

        rel = pos[col] - pos[row]
        sh = o3.spherical_harmonics(
            l=1,
            x=rel,
            normalize=True,
            normalization='component',
        )

        m_ij = self.tp_msg(h[col], sh)

        deg = scatter(torch.ones_like(row, dtype=m_ij.dtype), row, dim=0, dim_size=x.size(0), reduce="sum").clamp(min=1.0)
        m_i = scatter(m_ij, row, dim=0, dim_size=x.size(0), reduce="sum")
        m_i = m_i / deg.unsqueeze(-1)

        m_i = self.lin_msg(m_i)
        m_i = self.norm(m_i)

        x_out = x + m_i
        return x_out


class EquivariantAffinityModel(nn.Module):
    def __init__(self, max_z: int = 100, n_layers: int = 3, radius: float = 5.0):
        super().__init__()

        self.emb = nn.Embedding(max_z, 16)
        self.irreps_in = o3.Irreps("16x0e + 1x1o")

        self.lin_in = o3.Linear(
            irreps_in=self.irreps_in,
            irreps_out=IRREPS_NODE,
        )

        self.layers = nn.ModuleList(
            [EquivariantMPBlock() for _ in range(n_layers)]
        )

        self.lin_out = o3.Linear(
            irreps_in=IRREPS_NODE,
            irreps_out=IRREPS_OUT,
        )
        self.readout = nn.Sequential(
            nn.Linear(IRREPS_OUT.dim, HIDDEN),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN, 1),
        )
        self.radius = radius

    def forward(self, data):
        pos = data.pos
        z = data.z
        batch = data.batch
        edge_index = getattr(data, "edge_index", None)

        if edge_index is None:
            edge_index = build_radius_edge_index(pos, batch=batch, r=self.radius).to(pos.device)

        center = scatter(pos, batch, dim=0, reduce="mean")
        pos_rel = (pos - center[batch])

        scalars = self.emb(z)
        x_in = torch.cat([scalars, pos_rel], dim=-1)

        x = self.lin_in(x_in)

        for layer in self.layers:
            x = layer(x, pos_rel, edge_index)

        node_scalars = self.lin_out(x)
        g = scatter(node_scalars, batch, dim=0, reduce="mean")

        affinity = self.readout(g).squeeze(-1)
        return affinity


In [ ]:
class GuidedAffinityWrapper(nn.Module):
    """
    Wraps an affinity model for classifier-free guidance.
    - Uses true protein atomic numbers (pocket_z) instead of a sentinel.
    - Centers pocket coords once so they are in the same frame as the sampler.
    - Respects per-graph batching when multiple ligands are sampled in parallel.
    """
    def __init__(self, affinity_model, pocket_pos, pocket_z=None, device='cpu', z_pocket_val=99):
        super().__init__()
        self.affinity_model = affinity_model
        pocket_pos = pocket_pos.to(device)
        pocket_center = pocket_pos.mean(dim=0, keepdim=True)
        self.register_buffer("pocket_pos_centered", pocket_pos - pocket_center)
        if pocket_z is None:
            pocket_z = torch.full((pocket_pos.size(0),), z_pocket_val, dtype=torch.long, device=device)
        else:
            pocket_z = pocket_z.to(device).long()
        self.register_buffer("pocket_z", pocket_z)
        self.num_pocket_atoms = pocket_pos.size(0)

    def forward(self, ligand_pos, ligand_v, batch_ligand, batch_protein, protein_pos=None):
        device = ligand_pos.device
        num_graphs = batch_ligand.max().item() + 1

        atomic_numbers = trans.get_atomic_number_from_index(
            ligand_v.detach().cpu(), mode='add_aromatic'
        )
        z_lig = torch.tensor(atomic_numbers, dtype=torch.long, device=device)

        use_provided_protein = (
            protein_pos is not None
            and batch_protein is not None
            and protein_pos.size(0) == self.num_pocket_atoms * num_graphs
        )
        if use_provided_protein:
            pocket_pos = protein_pos
            pocket_batch = batch_protein
            z_pocket = self.pocket_z.repeat(num_graphs).to(device)
        else:
            pocket_pos = self.pocket_pos_centered.repeat(num_graphs, 1)
            pocket_batch = torch.arange(num_graphs, device=device).repeat_interleave(self.num_pocket_atoms)
            z_pocket = self.pocket_z.repeat(num_graphs).to(device)

        pos = torch.cat([ligand_pos, pocket_pos], dim=0)
        z = torch.cat([z_lig, z_pocket], dim=0)
        batch = torch.cat([batch_ligand, pocket_batch], dim=0)
        node_type = torch.cat([
            torch.ones(ligand_pos.size(0), dtype=torch.long, device=device),
            torch.zeros(pocket_pos.size(0), dtype=torch.long, device=device),
        ])

        data = Data(pos=pos, z=z, batch=batch, node_type=node_type)
        affinity = self.affinity_model(data)
        return -affinity


Usage (pseudo):

```python
aff_model = EquivariantAffinityModel(max_z=100).to(device)
aff_model.load_state_dict(torch.load('affinity_latest.pt')['model'])
wrapper = GuidedAffinityWrapper(
    affinity_model=aff_model,
    pocket_pos=data.protein_pos,
    pocket_z=data.protein_element,
    device=device,
)

# inside diffusion sampler, pass guidance_model=wrapper, guidance_scale>0
```
